In [ ]:
import json
from pathlib import Path

import pandas as pd
from tdc.benchmark_group import admet_group

predictions_dir = Path("tdc_splits/model_preds")
output_dir = Path("tdc_splits/metrics")

output_dir.mkdir(parents=True, exist_ok=True)

group = admet_group(path="data/")


all_metrics = []

for json_file in predictions_dir.glob("*.json"):

    print(f"\nProcessing: {json_file.name}")

    try:

        with open(json_file, "r") as f:
            predictions_list = json.load(f)


        Name = list(predictions_list[0].keys())[0]

        benchmark = group.get(Name)
        name = benchmark["name"]

        print(f"Benchmark: {name}")

        if len(predictions_list) == 1:

            predictions = {
                name: predictions_list[0][Name]
            }

            results = group.evaluate(predictions)

            print("Results:", results)

            # Extract metric
            metric_dict = results[name]

            row = {
                "data_name": name,
                "spearman": None,
                "mae": None,
                "std": None,
                "source_file": json_file.name
            }

            if "spearman" in metric_dict:
                row["spearman"] = float(metric_dict["spearman"])

            all_metrics.append(row)

        else:

            results = group.evaluate_many(predictions_list)

            print("Results:", results)

            metric_values = results[name]

            row = {
                "data_name": name,
                "spearman": None,
                "mae": None,
                "std": None,
                "source_file": json_file.name
            }

            # evaluate_many returns [MAE, STD]
            row["mae"] = float(metric_values[0])
            row["std"] = float(metric_values[1])

            all_metrics.append(row)

    except Exception as e:
        print(f"ERROR processing {json_file.name}: {e}")



metrics_df = pd.DataFrame(all_metrics)


output_file = output_dir / "all_metrics.csv"

metrics_df.to_csv(output_file, index=False)
print(f"Metrics saved to: {output_file}")

print(metrics_df)